In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

In [2]:
path = "model/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(path)
model = AutoModelForCausalLM.from_pretrained(path)

# 加载模型

In [3]:
p_model = PeftModel.from_pretrained(model, model_id="./chatbot/lora/checkpoint-2441/")
p_model.cuda()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
                (bas

```py
PeftModelForCausalLM(
  (base_model): LoraModel(...)
)
```
可以发现多了一层这个 LoRA Model 进行包装

# Inference


In [5]:
ipt = tokenizer("Human: {}\n{}".format("怎么学习线性代数？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to("cuda")
out = tokenizer.decode(p_model.generate(**ipt, max_length=256, do_sample=False)[0], skip_special_tokens=True)
print(out)

Human: 怎么学习线性代数？

Assistant: 学习线性代数是一个非常重要的数学技能，它在许多领域都有广泛的应用。以下是一些学习线性代数的建议：

1. 学习基础概念：线性代数的基本概念包括向量、矩阵、线性方程组、线性空间、线性组合、线性变换等。学习这些基本概念有助于理解线性代数的理论基础。

2. 学习向量和矩阵：向量和矩阵是线性代数中最基本的元素。学习向量和矩阵的运算，如加法、乘法、点积、叉积、矩阵乘法等，有助于理解线性代数的运算规则。

3. 学习线性方程组：线性方程组是线性代数中的一个重要概念。学习线性方程组的求解方法，如克莱姆法则、矩阵求逆、行列式等，有助于理解线性代数的解法。

4. 学习线性空间：线性空间是线性代数中的一个重要概念。学习线性空间的定义、基、维数、线性组合、线性变换


## 模型保存和合并

In [6]:
merge_model = p_model.merge_and_unload()
merge_model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

## 保存完整的模型

（如果只需要保存lora：`merge_model = p_model.merge_and_unload()`）

In [ ]:
save_path = "./chatbot/merge_model"
merged_model = p_model.merge_and_unload(save_path)

merge_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('./chatbot/merge_model/tokenizer_config.json',
 './chatbot/merge_model/special_tokens_map.json',
 './chatbot/merge_model/chat_template.jinja',
 './chatbot/merge_model/vocab.json',
 './chatbot/merge_model/merges.txt',
 './chatbot/merge_model/added_tokens.json',
 './chatbot/merge_model/tokenizer.json')

In [10]:
test_model = AutoModelForCausalLM.from_pretrained("./chatbot/merge_model")
test_model.cuda()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [11]:
ipt = tokenizer("Human: {}\n{}".format("怎么学习线性代数？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to("cuda")
out = tokenizer.decode(test_model.generate(**ipt, max_length=256, do_sample=False)[0], skip_special_tokens=True)
print(out)

Human: 怎么学习线性代数？

Assistant: 学习线性代数是一个非常重要的数学技能，它在许多领域都有广泛的应用。以下是一些学习线性代数的建议：

1. 学习基础概念：线性代数的基本概念包括向量、矩阵、线性方程组、线性空间、线性组合、线性变换等。学习这些基本概念有助于理解线性代数的理论基础。

2. 学习向量和矩阵：向量和矩阵是线性代数中最基本的元素。学习向量和矩阵的运算，如加法、乘法、点积、叉积、矩阵乘法等，有助于理解线性代数的运算规则。

3. 学习线性方程组：线性方程组是线性代数中的一个重要概念。学习线性方程组的求解方法，如克莱姆法则、矩阵求逆、行列式等，有助于理解线性代数的解法。

4. 学习线性空间：线性空间是线性代数中的一个重要概念。学习线性空间的定义、基、维数、线性组合、线性变换
